# The Error-State Kalman Filter for temporally consistent 6D Pose estimation

A per-frame 6D Pose estimator looks at every camera image on its own. It has no memory of previous frame, so its output is **not temporally consistent**: The pose jitters from frame to frame, it is sometimes completely wrong, it can flip between symmetric solutions, and it is missing whenever the object is not detected.

This notebook shows how an **Error-State Kalman Filter (ESKF)** turns such per-frame estimates into a smooth, robust and honest track. We build the filter from first principles in small steps, and test every piece before we use it.

It is written in the spirit of Roger Labbe's [*Kalman and Bayesian Filters in Python*](https://github.com/rlabbe/Kalman-and-Bayesian-Filters-in-Python) and follows the notation of

> J. Solà, *Quaternion kinematics for the error-state Kalman filter*, 2017. [arXiv:1711.02508](https://arxiv.org/abs/1711.02508)

**Contents**

1. The problem, and the idea of filtering
2. Why an *error-state* filter for poses?
3. A small quaternion toolbox
4. The data: a simulated per-frame pose estimator
5. True, nominal and error state
6. Prediction: a constant-velocity motion model
7. Correction: using one pose estimate
8. A first filter
9. Robustness: symmetry flips and outlier gating
10. Temporal consistency: results
11. Tuning: smoothness versus lag
12. Is the filter honest? NEES and NIS
13. Summary and references

Run the cells from top to bottom.

In [ ]:
%matplotlib inline
import os
from dataclasses import dataclass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
np.set_printoptions(precision=4, suppress=True)

## 1. The problem, and the idea of filtering

A per-frame estimator (for example a render-and-compare method such as FoundationPose) gives one pose $(\mathbf p_m, \mathbf q_m)$ of the object in the camera frame for every image. Four things go wrong from the point of view of temporal consistency:

| problem | what you see | cause |
|---|---|---|
| **jitter** | the pose shakes by millimetres and degrees, even when nothing moves | random error of each frame, e.g. from image noise and depth noise |
| **outliers** | a single frame is completely wrong | wrong match, wrong mask, bad depth |
| **symmetry flips** | the pose jumps by 180° and back | a near-symmetric object looks the same from both sides |
| **dropouts** | no pose for some frames | occlusion, turbidity, detection failure |

On top of that, the quality is not constant: estimates get worse with distance and in poor visibility.

**The idea of filtering.** A filter adds memory. At every frame it holds two opinions about the pose:

1. **Prediction:** "given where the object was and how it was moving, it should be *here* now".
2. **Measurement:** "the estimator says it is *there*".

The Kalman filter blends the two, weighted by how uncertain each one is. If the prediction is more certain than the measurement, the filter moves only part of the way towards the measurement, and the random part of each frame's error averages out over time. That is how jitter is reduced. The same machinery also lets us reject measurements that disagree too much with the prediction (outliers, flips), and keep predicting when measurements are missing.

**What a filter cannot do.** It removes *random* error, not *systematic* error: if every estimate is 2 cm too far away, the filtered track will be smoothly 2 cm too far away. And there is always a trade-off: trusting the prediction more gives a smoother track, but it reacts later to real motion (Section 11).

A filter is also **causal**: at frame $k$ it uses only the frames up to $k$, never later ones. That is exactly what you need for real-time tracking on a vehicle.

## 2. Why an *error-state* filter for poses?

A plain Kalman filter estimates the state directly. That is fine for position, but awkward for **orientation**:

* A unit quaternion has 4 numbers but only 3 degrees of freedom. A $4\times4$ covariance on it is singular in the direction that would violate $\|\mathbf q\|=1$, and after every update the quaternion leaves the unit sphere and must be renormalised, which the covariance knows nothing about.
* Euler angles have 3 numbers, but they have singularities (gimbal lock) and are strongly non-linear.

The ESKF splits the state in two:

| | **nominal state** $\mathbf x$ | **error state** $\delta\mathbf x$ |
|---|---|---|
| holds | the large signal, e.g. a quaternion | the small deviation from it |
| updated | non-linearly, without noise | linearly, carries all uncertainty |
| orientation | $\mathbf q$ (4 numbers, unit length) | $\delta\boldsymbol\theta\in\mathbb R^3$ (3 numbers, minimal) |

The covariance lives on the small error state, so it is $3\times3$ for orientation and never singular. The error stays close to zero, where the linearisation is accurate. And, as we will see in Section 7, a 6D pose measurement gets a very simple Jacobian in this formulation.

## 3. A small quaternion toolbox

We only need a handful of rotation operations. We build them one at a time and test each one.

Conventions used throughout:

* Hamilton quaternions, stored as $\mathbf q = [q_w, q_x, q_y, q_z]$.
* $\mathbf q$ rotates object-frame vectors into the camera frame: $\mathbf x_{cam} = \mathbf R(\mathbf q)\,\mathbf x_{obj}$.

### 3.1 The skew-symmetric matrix

$[\mathbf a]_\times$ turns a cross product into a matrix product: $[\mathbf a]_\times\mathbf b = \mathbf a\times\mathbf b$.

In [7]:
def skew(a):
    """[a]x such that skew(a) @ b == np.cross(a, b)."""
    x, y, z = a
    return np.array([[0.0, -z, y],
                     [z, 0.0, -x],
                     [-y, x, 0.0]])

In [8]:
a, b = np.array([1.0, 2.0, 3.0]), np.array([-0.5, 0.4, 2.0])
print("skew(a) @ b = ", skew(a) @ b)
print("np.cross(a, b) = ", np.cross(a, b))

skew(a) @ b =  [ 2.8 -3.5  1.4]
np.cross(a, b) =  [ 2.8 -3.5  1.4]


### 3.2 Quaternion product and conjugate

Composing two rotations is the quaternion product $\mathbf p\otimes\mathbf q$ (Solà eq. 12). The conjugate $\mathbf q^* = [q_w, -q_x, -q_y, -q_z]$ is the inverse rotation.

In [9]:
def q_mult(p, q):
    """Quaternion product p ⊗ q."""
    pw, px, py, pz = p
    qw, qx, qy, qz = q
    return np.array([pw*qw - px*qx - py*qy - pz*qz,
                     pw*qx + px*qw + py*qz - pz*qy,
                     pw*qy - px*qz + py*qw + pz*qx,
                     pw*qz + px*qy - py*qx + pz*qw])

def q_conj(q):
    """Conugate q*, the inverse of a unit quaternion."""
    return np.array([q[0], -q[1], -q[2], -q[3]])

In [10]:
q = np.array([0.8, 0.2, -0.4, 0.4]); q = q / np.linalg.norm(q)
p = np.array([0.6, 0.0, 0.8, 0.0]); p = p / np.linalg.norm(p)
print("q ⊗ q* :", q_mult(q, q_conj(q)), " (identity)")
print("p ⊗ q  :", q_mult(p, q))
print("q ⊗ p  :", q_mult(q, p), " (different: rotations do not commute)")

q ⊗ q* : [1. 0. 0. 0.]  (identity)
p ⊗ q  : [0.8  0.44 0.4  0.08]
q ⊗ p  : [ 0.8 -0.2  0.4  0.4]  (different: rotations do not commute)


### 3.3 Exp and Log: rotation vectors and quaternions

A rotation by angle $\theta$ about unit axis $\mathbf u$ can be written as the 3-vector $\boldsymbol\theta = \theta\mathbf u$. The exponential and logarithmic maps convert between the two forms (Solà eqs. 101, 105):

$$
\mathrm{Exp}(\theta\mathbf u) = \begin{bmatrix}\cos(\theta/2)\\ \mathbf u\sin(\theta/2)\end{bmatrix},
\qquad
\mathrm{Log}(\mathbf q) = \theta\mathbf u .
$$

These two maps are the key to the ESKF: the orientation *error* will be a small 3-vector $\delta\boldsymbol\theta$, and Exp turns it into a quaternion we can apply.

In [11]:
def q_normalize(q):
    return q / np.linalg.norm(q)

def q_exp(rotvec):
    """Rotation vector θu -> unit quaternion."""
    rotvec = np.asarray(rotvec, dtype=float)
    theta = np.linalg.norm(rotvec)
    if theta < 1e-10:
        return q_normalize(np.array([1.0, *(0.5 * rotvec)]))
    u = rotvec / theta
    return np.array([np.cos(theta / 2.0), *(np.sin(theta / 2.0) * u)])

def q_log(q):
    """Unit quaternion -> rotation vector θu."""
    q = np.asarray(q, dtype=float)
    if q[0] < 0:
        q = -q
    v = q[1:]
    norm_v = np.linalg.norm(v)
    if norm_v < 1e-10:
        return 2.0 * v
    return 2.0 * np.arctan2(norm_v, q[0]) * (v / norm_v)

In [12]:
theta = np.array([0.0, 0.0, np.pi / 2])    # 90 deg about z-axis
print("Exp(90° about z) :", q_exp(theta))
print("Log(Exp(θ)) == θ :", np.allclose(q_log(q_exp(theta)), theta))
print("Log(-q) == Log(q):", np.allclose(q_log(-q_exp(theta)), theta))

Exp(90° about z) : [0.7071 0.     0.     0.7071]
Log(Exp(θ)) == θ : True
Log(-q) == Log(q): True


In [14]:
def q_to_R(q):
    """Rotation matrix with x_cam = R @ x_obj."""
    w, x, y, z = q
    return np.array([[1 - 2*(y*y + z*z), 2*(x*y - w*z), 2*(x*z + w*y)],
                     [2*(x*y + w*z), 1 - 2*(x*x + z*z), 2*(y*z - w*x)],
                     [2*(x*z - w*y), 2*(y*z + w*x), 1 - 2*(x*x + y*y)]])

In [16]:
R = q_to_R(q_exp([0.0, 0.0, np.pi / 2]))
print("x-axis rotated 90° about z:", R @ np.array([1.0, 0.0, 0.0]), " (should be the y-axis)")
print("R Rᵀ == I:", np.allclose(R @ R.T, np.eye(3)))

x-axis rotated 90° about z: [0. 1. 0.]  (should be the y-axis)
R Rᵀ == I: True


### 3.5 "Plus" and "minus" for rotations

Rotations cannot be added and subtracted like vectors, but Exp and Log give us the equivalent operations. We use a **local** perturbation, applied in the object frame (on the right):

$$
\mathbf q \boxplus \delta\boldsymbol\theta = \mathbf q \otimes \mathrm{Exp}(\delta\boldsymbol\theta),
\qquad
\mathbf q_1 \boxminus \mathbf q_0 = \mathrm{Log}(\mathbf q_0^{*} \otimes \mathbf q_1).
$$

$\boxplus$ applies a small rotation. $\boxminus$ answers "which rotation takes $\mathbf q_0$ to $\mathbf q_1$?". Its length $\|\mathbf q_1\boxminus\mathbf q_0\|$ is the angle between two orientations, which we will use as the rotation error.

In [18]:
def q_boxplus(q, dtheta):
    """q ⊞ δθ : apply a small object-frame rotation δθ to q."""
    return q_normalize(q_mult(q, q_exp(dtheta)))

def q_boxminus(q1, q0):
    """q1 ⊟ q0 : the object-frame rotation that takes q0 to q1."""
    return q_log(q_mult(q_conj(q0), q1))

def rotation_angle_deg(q1, q0):
    """Angle between two orientations, in degrees."""
    return np.degrees(np.linalg.norm(q_boxminus(q1, q0)))

In [19]:
q0 = q_exp([0.3, -0.2, 0.5])
dtheta = np.array([0.01, -0.02, 0.03])
q1 = q_boxplus(q0, dtheta)
print("(q0 ⊞ δθ) ⊟ q0 :", q_boxminus(q1, q0), " (should be close to δθ)")
print("Angle between :", round(rotation_angle_deg(q1, q0), 3), "deg =", round(np.degrees(np.linalg.norm(dtheta)), 3), "deg")

(q0 ⊞ δθ) ⊟ q0 : [ 0.01 -0.02  0.03]  (should be close to δθ)
Angle between : 2.144 deg = 2.144 deg


## 4. The data: a simulated per-frame pose estimator

To see what the filter does, we need the ground truth, so we simulate. The scene: a camera on an underwater vehicle observes a static object from about $1.5$–$2.5$ m while the vehicle moves. The *relative* pose of the object in the camera frame therefore changes smoothly. A per-frame estimator runs at 15 Hz and has all the problems from Section 1:

| effect | in the simulation |
|---|---|
| jitter | Gaussian noise, $\sigma_p \approx 1$ cm and $\sigma_\theta\approx 1.5°$ at 1.5 m, growing with distance |
| poor visibility | between 14 s and 20 s the noise is 3× larger ("turbid water") |
| outliers | 3% of frames are a random, completely wrong pose |
| symmetry flips | 5% of frames are flipped by 180° about the object's $z$-axis, a symmetry of the object |
| dropouts | 5% of frames missing at random, plus an occlusion from 26 s to 29 s |

Like a good estimator, it also reports its own uncertainty $\sigma_p,\sigma_\theta$ for every frame. For outliers and flips this reported uncertainty is as small as usual, because the estimator does not know it is wrong. The first second is kept clean so the filter can start on a good estimate.

First the configuration and the true motion: